# Flipkart Customer Support — Machine Learning Submission

## Project Summary
This notebook implements the Machine Learning component of the Flipkart Customer Support project.

**Objective:** Predict whether a customer is satisfied based on available customer-support interaction features.

**Target definition:**  
- CSAT Score 4–5 → **Satisfied (1)**
- CSAT Score 1–3 → **Not Satisfied (0)**

The notebook is intentionally kept separate from the EDA submission and follows the same professional, concise presentation style.

## GitHub Link
**GitHub Repository:** Add your repository link here before submission.

## Problem Statement
Customer support teams need to understand which interactions are likely to result in customer satisfaction. The objective is to build classification models that predict customer satisfaction from support-channel, issue, agent, shift, and response-time information.

### Business Objective
Build and compare classification models and identify the model that provides the strongest predictive performance for customer satisfaction.

## General Guidelines
- Use the supplied Flipkart Customer Support dataset.
- Keep preprocessing inside the ML pipeline where appropriate.
- Use a stratified train/test split.
- Compare multiple models using the same test data.
- Report Accuracy, Precision, Recall, F1 Score and ROC-AUC.
- Use cross-validation and hyperparameter tuning where applicable.
- Interpret the selected model from a business perspective.
- Avoid data leakage by fitting preprocessing only on training data.

# 1. Know Your Data
This section loads the dataset and checks its basic structure before model preparation.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    RandomizedSearchCV
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

DATA_PATH = "Customer_support_data.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("Data Type"))

print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
missing_summary = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Missing Percentage": (df.isna().mean() * 100).round(2)
}).sort_values("Missing Values", ascending=False)

display(missing_summary)

# 2. Understanding Your Variables
The model uses operational support variables that are available before or during the support interaction, along with engineered response-time and time-of-day features.

In [ ]:
variable_description = pd.DataFrame({
    "Variable": [
        "channel_name", "category", "Sub-category",
        "Tenure Bucket", "Agent Shift",
        "response_time_minutes", "report_hour",
        "report_day", "report_month"
    ],
    "Role": [
        "Support channel",
        "Main issue category",
        "Issue sub-category",
        "Agent tenure group",
        "Agent working shift",
        "Time taken to respond",
        "Hour of issue reporting",
        "Day of month of issue reporting",
        "Month of issue reporting"
    ]
})

display(variable_description)

# 3. Data Wrangling
The raw data is prepared for modelling. Extremely high-missingness fields are excluded from the core model, date fields are parsed, and useful time/response features are created.

In [ ]:
df_clean = df.copy()

# Remove duplicate records
df_clean = df_clean.drop_duplicates().copy()

# Parse date/time columns
date_columns = [
    "order_date_time",
    "Issue_reported at",
    "issue_responded",
    "Survey_response_Date"
]

for col in date_columns:
    if col in df_clean.columns:
        df_clean[col] = pd.to_datetime(
            df_clean[col],
            errors="coerce",
            dayfirst=True,
            format="mixed"
        )

# Clean text columns
text_columns = df_clean.select_dtypes(include=["object", "string"]).columns
for col in text_columns:
    df_clean[col] = df_clean[col].str.strip()

# Drop extremely high-missingness columns for the core model
high_missing_columns = [
    "connected_handling_time",
    "Customer_City",
    "Product_category",
    "Item_price",
    "order_date_time"
]

df_clean = df_clean.drop(
    columns=[c for c in high_missing_columns if c in df_clean.columns]
)

# Fill selected text fields
if "Customer Remarks" in df_clean.columns:
    df_clean["Customer Remarks"] = df_clean["Customer Remarks"].fillna("No Remarks")

if "Order_id" in df_clean.columns:
    df_clean["Order_id"] = df_clean["Order_id"].fillna("Not Available")

# Engineer response time
if "Issue_reported at" in df_clean.columns and "issue_responded" in df_clean.columns:
    df_clean["response_time_minutes"] = (
        df_clean["issue_responded"] - df_clean["Issue_reported at"]
    ).dt.total_seconds() / 60

    df_clean.loc[
        (df_clean["response_time_minutes"] < 0) |
        (df_clean["response_time_minutes"] > 7 * 24 * 60),
        "response_time_minutes"
    ] = np.nan

    df_clean["report_hour"] = df_clean["Issue_reported at"].dt.hour
    df_clean["report_day"] = df_clean["Issue_reported at"].dt.day
    df_clean["report_month"] = df_clean["Issue_reported at"].dt.month

print("Cleaned dataset shape:", df_clean.shape)
display(df_clean.head())

## ML Dataset Preparation
The target is derived directly from CSAT Score. The feature list is kept consistent with the original ML implementation.

In [ ]:
# Target: CSAT 4-5 = Satisfied, CSAT 1-3 = Not Satisfied
model_df = df_clean.copy()
model_df["satisfied"] = (model_df["CSAT Score"] >= 4).astype(int)

feature_columns = [
    "channel_name",
    "category",
    "Sub-category",
    "Tenure Bucket",
    "Agent Shift",
    "response_time_minutes",
    "report_hour",
    "report_day",
    "report_month"
]

missing_features = [col for col in feature_columns if col not in model_df.columns]
print("Missing features:", missing_features)

X = model_df[feature_columns].copy()
y = model_df["satisfied"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nTarget distribution:")
display(y.value_counts().rename({0: "Not Satisfied", 1: "Satisfied"}).to_frame("Count"))

# 4. Feature Engineering & Data Pre-processing
Categorical variables are imputed and one-hot encoded. Numerical variables are median-imputed and standardized. All transformations are placed inside a `ColumnTransformer` so that preprocessing is learned only from the training data.

In [ ]:
categorical_features = X.select_dtypes(
    include=["object", "string"]
).columns.tolist()

numerical_features = [
    col for col in X.columns
    if col not in categorical_features
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        ),
        (
            "numerical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numerical_features
        )
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

# 5. ML Model Implementation

## Model 1 — Logistic Regression
Logistic Regression provides an interpretable linear classification baseline.

In [ ]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

log_pred = logistic_model.predict(X_test)
log_prob = logistic_model.predict_proba(X_test)[:, 1]

log_metrics = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test, log_pred),
    "Precision": precision_score(y_test, log_pred),
    "Recall": recall_score(y_test, log_pred),
    "F1 Score": f1_score(y_test, log_pred),
    "ROC-AUC": roc_auc_score(y_test, log_prob)
}

display(pd.DataFrame([log_metrics]).round(4))

## Model 2 — Random Forest
Random Forest captures nonlinear relationships and interactions between customer-support features.

In [ ]:
random_forest_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=150,
        max_depth=16,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

random_forest_model.fit(X_train, y_train)

rf_pred = random_forest_model.predict(X_test)
rf_prob = random_forest_model.predict_proba(X_test)[:, 1]

rf_metrics = {
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_test, rf_pred),
    "Precision": precision_score(y_test, rf_pred),
    "Recall": recall_score(y_test, rf_pred),
    "F1 Score": f1_score(y_test, rf_pred),
    "ROC-AUC": roc_auc_score(y_test, rf_prob)
}

display(pd.DataFrame([rf_metrics]).round(4))

# 6. Model Evaluation & Comparison
Both models are evaluated on the same held-out test set using the same metrics.

In [ ]:
comparison = pd.DataFrame(
    [log_metrics, rf_metrics]
).set_index("Model")

display(comparison.round(4))

best_model_name = comparison["F1 Score"].idxmax()
print("Model with the highest F1 Score:", best_model_name)

In [ ]:
metric_plot = comparison[
    ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]
]

ax = metric_plot.plot(
    kind="bar",
    figsize=(10, 6)
)
ax.set_title("Model Performance Comparison")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Cross-Validation
Stratified cross-validation is used to assess model stability beyond a single train/test split.

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results = []

for name, model in [
    ("Logistic Regression", logistic_model),
    ("Random Forest", random_forest_model)
]:
    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    cv_results.append({
        "Model": name,
        "CV Accuracy": scores["test_accuracy"].mean(),
        "CV Precision": scores["test_precision"].mean(),
        "CV Recall": scores["test_recall"].mean(),
        "CV F1": scores["test_f1"].mean(),
        "CV ROC-AUC": scores["test_roc_auc"].mean()
    })

cv_comparison = pd.DataFrame(cv_results).set_index("Model")
display(cv_comparison.round(4))

## Hyperparameter Tuning — Logistic Regression
RandomizedSearchCV is used to test a small, controlled parameter space while keeping the original preprocessing pipeline.

In [ ]:
logistic_param_distributions = {
    "model__C": np.logspace(-2, 1, 8),
    "model__solver": ["liblinear", "lbfgs"]
}

logistic_search = RandomizedSearchCV(
    estimator=Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ]),
    param_distributions=logistic_param_distributions,
    n_iter=6,
    scoring="f1",
    cv=5,
    random_state=42,
    n_jobs=-1
)

logistic_search.fit(X_train, y_train)

tuned_log_pred = logistic_search.predict(X_test)
tuned_log_prob = logistic_search.predict_proba(X_test)[:, 1]

tuned_log_metrics = {
    "Model": "Tuned Logistic Regression",
    "Accuracy": accuracy_score(y_test, tuned_log_pred),
    "Precision": precision_score(y_test, tuned_log_pred),
    "Recall": recall_score(y_test, tuned_log_pred),
    "F1 Score": f1_score(y_test, tuned_log_pred),
    "ROC-AUC": roc_auc_score(y_test, tuned_log_prob)
}

print("Best parameters:", logistic_search.best_params_)
display(pd.DataFrame([tuned_log_metrics]).round(4))

## Hyperparameter Tuning — Random Forest
A controlled randomized search is used for the Random Forest hyperparameters.

In [ ]:
rf_param_distributions = {
    "model__n_estimators": [100, 150, 200, 250],
    "model__max_depth": [10, 12, 16, 20, None],
    "model__min_samples_leaf": [2, 5, 8, 10],
    "model__max_features": ["sqrt", "log2"]
}

rf_search = RandomizedSearchCV(
    estimator=Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]),
    param_distributions=rf_param_distributions,
    n_iter=8,
    scoring="f1",
    cv=5,
    random_state=42,
    n_jobs=-1
)

rf_search.fit(X_train, y_train)

tuned_rf_pred = rf_search.predict(X_test)
tuned_rf_prob = rf_search.predict_proba(X_test)[:, 1]

tuned_rf_metrics = {
    "Model": "Tuned Random Forest",
    "Accuracy": accuracy_score(y_test, tuned_rf_pred),
    "Precision": precision_score(y_test, tuned_rf_pred),
    "Recall": recall_score(y_test, tuned_rf_pred),
    "F1 Score": f1_score(y_test, tuned_rf_pred),
    "ROC-AUC": roc_auc_score(y_test, tuned_rf_prob)
}

print("Best parameters:", rf_search.best_params_)
display(pd.DataFrame([tuned_rf_metrics]).round(4))

# 7. Final Model Comparison & Selection
The original two models and their tuned versions are compared using the same held-out test set. F1 Score is used as the primary selection metric because it balances Precision and Recall.

In [ ]:
final_comparison = pd.DataFrame([
    log_metrics,
    rf_metrics,
    tuned_log_metrics,
    tuned_rf_metrics
]).set_index("Model")

display(final_comparison.round(4))

final_model_name = final_comparison["F1 Score"].idxmax()
print("Selected final model:", final_model_name)

In [ ]:
model_predictions = {
    "Logistic Regression": log_pred,
    "Random Forest": rf_pred,
    "Tuned Logistic Regression": tuned_log_pred,
    "Tuned Random Forest": tuned_rf_pred
}

best_pred = model_predictions[final_model_name]

print(classification_report(
    y_test,
    best_pred,
    target_names=["Not Satisfied", "Satisfied"]
))

## Confusion Matrix
The confusion matrix shows correct and incorrect predictions for the selected final model.

In [ ]:
cm = confusion_matrix(y_test, best_pred)

plt.figure(figsize=(6, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Not Satisfied", "Satisfied"]
)
disp.plot(values_format="d")
plt.title(f"Confusion Matrix — {final_model_name}")
plt.tight_layout()
plt.show()

## Feature Importance
For business interpretability, feature importance is extracted when the selected model is a Random Forest. If Logistic Regression is selected, the notebook reports that tree-based feature importance is not applicable to the selected model.

In [ ]:
if "Random Forest" in final_model_name:
    selected_search = rf_search if final_model_name == "Tuned Random Forest" else None

    if selected_search is not None:
        selected_pipeline = selected_search.best_estimator_
    else:
        selected_pipeline = random_forest_model

    rf_preprocessor = selected_pipeline.named_steps["preprocessor"]
    rf_estimator = selected_pipeline.named_steps["model"]

    feature_names = rf_preprocessor.get_feature_names_out()

    importance = pd.Series(
        rf_estimator.feature_importances_,
        index=feature_names
    ).sort_values(ascending=False).head(15)

    display(importance.to_frame("Importance").round(4))

    plt.figure(figsize=(10, 6))
    importance.sort_values().plot.barh()
    plt.title("Top 15 Random Forest Feature Importances")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()
else:
    print(
        "The selected final model is Logistic Regression. "
        "Random Forest feature importance is therefore not used for the selected model."
    )

# 8. Business Impact
The model can support customer-service operations by identifying patterns associated with customer satisfaction.

Potential uses:
- Prioritize interactions that may need additional attention.
- Identify operational factors associated with lower satisfaction.
- Support agent/shift performance analysis.
- Help managers focus improvement efforts on measurable service factors.

**Important:** The model is a decision-support tool and should not replace human judgment.

## Model Saving
The selected trained model can be saved for later use.

In [ ]:
import joblib

model_objects = {
    "Logistic Regression": logistic_model,
    "Random Forest": random_forest_model,
    "Tuned Logistic Regression": logistic_search.best_estimator_,
    "Tuned Random Forest": rf_search.best_estimator_
}

final_model = model_objects[final_model_name]

joblib.dump(
    final_model,
    "flipkart_customer_satisfaction_model.joblib"
)

print("Saved final model:", final_model_name)
print("File: flipkart_customer_satisfaction_model.joblib")

## Model Loading & Sanity Check
Reload the saved model and generate predictions on a small test sample.

In [ ]:
loaded_model = joblib.load(
    "flipkart_customer_satisfaction_model.joblib"
)

sample_predictions = loaded_model.predict(X_test.head(5))

display(pd.DataFrame({
    "Actual": y_test.head(5).values,
    "Predicted": sample_predictions
}))

# Conclusion
This ML notebook builds a customer-satisfaction classification solution using the Flipkart Customer Support dataset.

Two models were implemented and compared:
1. **Logistic Regression**
2. **Random Forest**

Both models were evaluated using Accuracy, Precision, Recall, F1 Score and ROC-AUC. Cross-validation and hyperparameter tuning were also performed. The final model is selected using the highest F1 Score on the held-out test data.

The model can provide useful decision-support signals for improving customer-support operations.

## Future Work
- Add richer historical customer interaction features.
- Include product/order information if reliable and sufficiently complete.
- Test additional advanced models if required.
- Perform deeper calibration and threshold analysis.
- Deploy the selected model through a monitoring-ready application.